In [1]:
print("Here begins the Bayesian networks notebook.")

Here begins the Bayesian networks notebook.


In [1]:
import pandas as pd

# Load data, display first few rows
pd.set_option("display.max_columns", None)
df_bn = pd.read_csv("../../data/processed/mental_health_tech_clean.csv")

df_bn.head()

,Age,Gender,Country,state,self_employed,family_history,treatment,work_interfere,no_employees,remote_work,tech_company,benefits,care_options,wellness_program,seek_help,anonymity,leave,mental_health_consequence,phys_health_consequence,coworkers,supervisor,mental_health_interview,phys_health_interview,mental_vs_physical,obs_consequence
0,37.0,female,United States,IL,NaN,0,1,Often,6-25,0,1,Yes,Not sure,No,Yes,Yes,Somewhat easy,No,No,Some of them,Yes,No,Maybe,Yes,0
1,44.0,male,United States,IN,NaN,0,0,Rarely,More than 1000,0,0,Don't know,No,Don't know,Don't know,Don't know,Don't know,Maybe,No,No,No,No,No,Don't know,0
2,32.0,male,Canada,NaN,NaN,0,0,Rarely,6-25,0,1,No,No,No,No,Don't know,Somewhat difficult,No,No,Yes,Yes,Yes,Yes,No,0
3,31.0,male,United Kingdom,NaN,NaN,1,1,Often,26-100,0,1,No,Yes,No,No,No,Somewhat difficult,Yes,Yes,Some of them,No,Maybe,Maybe,No,1
4,31.0,male,United States,TX,NaN,0,0,Never,100-500,1,1,Yes,No,Don't know,Don't know,Don't know,Don't know,No,No,Some of them,Yes,Yes,Yes,Don't know,0


In [2]:
# Inspect dataset structure
# Check that the BN dataset contains the expected variables
# Identify missing values before structure learning
print(df_bn.shape)
print(df_bn.dtypes)
print("\nMissing values per column:")
print(df_bn.isna().sum().sort_values(ascending=False))

(1259, 25)
Age                          float64
Gender                           str
Country                          str
state                            str
self_employed                float64
family_history                 int64
treatment                      int64
work_interfere                   str
no_employees                     str
remote_work                    int64
tech_company                   int64
benefits                         str
care_options                     str
wellness_program                 str
seek_help                        str
anonymity                        str
leave                            str
mental_health_consequence        str
phys_health_consequence          str
coworkers                        str
supervisor                       str
mental_health_interview          str
phys_health_interview            str
mental_vs_physical               str
obs_consequence                int64
dtype: object

Missing values per column:
state                 

In [3]:
# Convert all columns to string / categorical-style values
# pgmpy structure learning for discrete BNs works with discrete states
# Missing values are preserved as NaN
for col in df_bn.columns:
    df_bn[col] = df_bn[col].astype("string")

print(df_bn.dtypes)

Age                          string
Gender                       string
Country                      string
state                        string
self_employed                string
family_history               string
treatment                    string
work_interfere               string
no_employees                 string
remote_work                  string
tech_company                 string
benefits                     string
care_options                 string
wellness_program             string
seek_help                    string
anonymity                    string
leave                        string
mental_health_consequence    string
phys_health_consequence      string
coworkers                    string
supervisor                   string
mental_health_interview      string
phys_health_interview        string
mental_vs_physical           string
obs_consequence              string
dtype: object


In [5]:
# Reduce very high-cardinality variables
# Bin Age into 5-year bins to reduce sparsity and improve structure learning
df_bn["Age_binned"] = pd.cut(
    pd.to_numeric(df_bn["Age"], errors="coerce"),
    bins=[0, 25, 35, 45, 60, 100],
    labels=["18-25", "26-35", "36-45", "46-60", "60+"]
).astype("string")

df_bn = df_bn.drop(columns=["Age"])

# Country and State can also create sparse CPDs and unstable structures
df_bn_model = df_bn.drop(columns=["Country", "state"], errors="ignore").copy()

print(df_bn_model.shape)
print(df_bn_model.columns.tolist())

(1259, 23)
['Gender', 'self_employed', 'family_history', 'treatment', 'work_interfere', 'no_employees', 'remote_work', 'tech_company', 'benefits', 'care_options', 'wellness_program', 'seek_help', 'anonymity', 'leave', 'mental_health_consequence', 'phys_health_consequence', 'coworkers', 'supervisor', 'mental_health_interview', 'phys_health_interview', 'mental_vs_physical', 'obs_consequence', 'Age_binned']


In [6]:
# Check unique states per variable
# Useful for spotting overly sparse variables before structure learning
cardinality = pd.DataFrame({
    "Variable": df_bn_model.columns,
    "Num_States": [df_bn_model[col].nunique(dropna=True) for col in df_bn_model.columns]
}).sort_values("Num_States", ascending=False)

cardinality

,Variable,Num_States
5,no_employees,6
22,Age_binned,5
13,leave,5
4,work_interfere,4
10,wellness_program,3
8,benefits,3
12,anonymity,3
18,mental_health_interview,3
15,phys_health_consequence,3
17,supervisor,3


In [7]:
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

# Imports for Bayesian Network structure learning and inference
from pgmpy.estimators import HillClimbSearch, ExpertKnowledge, BayesianEstimator
from pgmpy.models import DiscreteBayesianNetwork
from pgmpy.inference import VariableElimination

In [8]:
# Learn an unconstrained BN structure using Hill-Climb Search
# - Hill-Climb Search is a score-based structure learning method
# - bic-d is a discrete BIC score, appropriate for categorical data
hc = HillClimbSearch(df_bn_model)

learned_dag = hc.estimate(
    scoring_method="bic-d",
    max_indegree=3,
    show_progress=False
)

print("Number of nodes:", len(learned_dag.nodes()))
print("Number of edges:", len(learned_dag.edges()))
print("\nLearned edges:")
print(sorted(learned_dag.edges()))

Number of nodes: 23
Number of edges: 41

Learned edges:
[('Gender', 'family_history'), ('Gender', 'leave'), ('Gender', 'no_employees'), ('Gender', 'obs_consequence'), ('Gender', 'seek_help'), ('Gender', 'supervisor'), ('Gender', 'tech_company'), ('Gender', 'treatment'), ('benefits', 'care_options'), ('care_options', 'anonymity'), ('mental_health_consequence', 'mental_health_interview'), ('mental_health_consequence', 'mental_vs_physical'), ('mental_health_consequence', 'phys_health_consequence'), ('mental_health_interview', 'phys_health_interview'), ('no_employees', 'self_employed'), ('seek_help', 'benefits'), ('seek_help', 'wellness_program'), ('self_employed', 'remote_work'), ('supervisor', 'coworkers'), ('supervisor', 'mental_health_consequence'), ('work_interfere', 'Age_binned'), ('work_interfere', 'Gender'), ('work_interfere', 'anonymity'), ('work_interfere', 'benefits'), ('work_interfere', 'care_options'), ('work_interfere', 'coworkers'), ('work_interfere', 'family_history'), ('wo

In [9]:
# Convert learned structure into a discrete Bayesian Network
bn_unconstrained = DiscreteBayesianNetwork(learned_dag.edges())

# Fit CPDs using Bayesian parameter estimation
# BayesianEstimator is preferred because it smooths sparse probability tables
bn_unconstrained.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Model check:", bn_unconstrained.check_model())

Model check: True


In [10]:
# Inspect learned CPDs for key outcome variables
# These are the two most important variables for discussion
for var in ["treatment", "work_interfere"]:
    if var in bn_unconstrained.nodes():
        print(f"\nCPD for {var}:")
        print(bn_unconstrained.get_cpds(var))


CPD for treatment:
+----------------+-----+---------------------------+
| Gender         | ... | Gender(male)              |
+----------------+-----+---------------------------+
| work_interfere | ... | work_interfere(Sometimes) |
+----------------+-----+---------------------------+
| treatment(0)   | ... | 0.2582903463522476        |
+----------------+-----+---------------------------+
| treatment(1)   | ... | 0.7417096536477524        |
+----------------+-----+---------------------------+

CPD for work_interfere:
+---------------------------+----------+
| work_interfere(Never)     | 0.214428 |
+---------------------------+----------+
| work_interfere(Often)     | 0.145771 |
+---------------------------+----------+
| work_interfere(Rarely)    | 0.174627 |
+---------------------------+----------+
| work_interfere(Sometimes) | 0.465174 |
+---------------------------+----------+


In [11]:
# Perform inference on the unconstrained BN
infer_unconstrained = VariableElimination(bn_unconstrained) # type: ignore

# Query 1:
# Probability of treatment given family history and frequent work interference
q1 = infer_unconstrained.query(
    variables=["treatment"],
    evidence={ # type: ignore
        "family_history": "1",
        "work_interfere": "Often"
    },
    show_progress=False
)

print(q1)

+--------------+------------------+
| treatment    |   phi(treatment) |
+==============+==================+
| treatment(0) |           0.1510 |
+--------------+------------------+
| treatment(1) |           0.8490 |
+--------------+------------------+


In [12]:
# Query 2:
# Probability of work interference given treatment and poor workplace support
q2 = infer_unconstrained.query(
    variables=["work_interfere"],
    evidence={ # type: ignore
        "treatment": "1",
        "benefits": "No",
        "seek_help": "No"
    },
    show_progress=False
)

print(q2)

+---------------------------+-----------------------+
| work_interfere            |   phi(work_interfere) |
+===========================+=======================+
| work_interfere(Never)     |                0.0391 |
+---------------------------+-----------------------+
| work_interfere(Often)     |                0.2532 |
+---------------------------+-----------------------+
| work_interfere(Rarely)    |                0.1548 |
+---------------------------+-----------------------+
| work_interfere(Sometimes) |                0.5530 |
+---------------------------+-----------------------+


In [13]:
# Query 3:
# Compare treatment probabilities under different policy conditions
q3_yes = infer_unconstrained.query(
    variables=["treatment"],
    evidence={ # type: ignore
        "care_options": "Yes",
        "seek_help": "Yes"
    },
    show_progress=False
)

q3_no = infer_unconstrained.query(
    variables=["treatment"],
    evidence={ # type: ignore
        "care_options": "No",
        "seek_help": "No"
    },
    show_progress=False
)

print("Treatment distribution with supportive care options:")
print(q3_yes)

print("\nTreatment distribution with poor care options:")
print(q3_no)

Treatment distribution with supportive care options:
+--------------+------------------+
| treatment    |   phi(treatment) |
+==============+==================+
| treatment(0) |           0.3326 |
+--------------+------------------+
| treatment(1) |           0.6674 |
+--------------+------------------+

Treatment distribution with poor care options:
+--------------+------------------+
| treatment    |   phi(treatment) |
+==============+==================+
| treatment(0) |           0.3667 |
+--------------+------------------+
| treatment(1) |           0.6333 |
+--------------+------------------+


In [14]:
# Build a second BN with simple expert constraints
# This version is often better for a thesis because it improves causal plausibility
# by preventing obviously implausible directions.

forbidden_edges = [
    # Outcomes should not cause background variables
    ("treatment", "Age_binned"),
    ("treatment", "Gender"),
    ("treatment", "family_history"),
    ("work_interfere", "Age_binned"),
    ("work_interfere", "Gender"),
    ("work_interfere", "family_history"),

    # Workplace policy variables should not be caused by outcomes
    ("treatment", "benefits"),
    ("treatment", "seek_help"),
    ("treatment", "wellness_program"),
    ("work_interfere", "benefits"),
    ("work_interfere", "seek_help"),
    ("work_interfere", "wellness_program"),
]

expert_knowledge = ExpertKnowledge(
    forbidden_edges=forbidden_edges
)

hc_restricted = HillClimbSearch(df_bn_model)

learned_dag_restricted = hc_restricted.estimate(
    scoring_method="bic-d",
    expert_knowledge=expert_knowledge,
    max_indegree=3,
    show_progress=False
)

print("Restricted BN edges:")
print(sorted(learned_dag_restricted.edges()))

Restricted BN edges:
[('Gender', 'Age_binned'), ('Gender', 'anonymity'), ('Gender', 'benefits'), ('Gender', 'family_history'), ('Gender', 'leave'), ('Gender', 'no_employees'), ('Gender', 'obs_consequence'), ('Gender', 'seek_help'), ('Gender', 'supervisor'), ('Gender', 'tech_company'), ('Gender', 'treatment'), ('Gender', 'wellness_program'), ('Gender', 'work_interfere'), ('anonymity', 'care_options'), ('benefits', 'seek_help'), ('care_options', 'benefits'), ('family_history', 'work_interfere'), ('mental_health_consequence', 'mental_health_interview'), ('mental_health_consequence', 'mental_vs_physical'), ('mental_health_consequence', 'phys_health_consequence'), ('mental_health_interview', 'phys_health_interview'), ('no_employees', 'self_employed'), ('seek_help', 'wellness_program'), ('self_employed', 'benefits'), ('self_employed', 'remote_work'), ('supervisor', 'coworkers'), ('supervisor', 'mental_health_consequence'), ('work_interfere', 'anonymity'), ('work_interfere', 'care_options'), 

In [15]:
# Fit the restricted BN
bn_restricted = DiscreteBayesianNetwork(learned_dag_restricted.edges())

bn_restricted.fit(
    df_bn_model,
    estimator=BayesianEstimator,
    prior_type="BDeu",
    equivalent_sample_size=10
)

print("Restricted model check:", bn_restricted.check_model())

Restricted model check: True


In [16]:
# Compare key parents of treatment and work_interfere
# This is useful for interpretation in the thesis
for target in ["treatment", "work_interfere"]:
    if target in bn_restricted.nodes():
        print(f"\nParents of {target} in restricted BN:")
        print(list(bn_restricted.predecessors(target)))


Parents of treatment in restricted BN:
['Gender', 'work_interfere']

Parents of work_interfere in restricted BN:
['Gender', 'family_history']


In [17]:
# Inference with restricted BN
infer_restricted = VariableElimination(bn_restricted) # type: ignore

# Query 4:
# Probability of treatment given family history and supportive care options
q4 = infer_restricted.query(
    variables=["treatment"],
    evidence={ # type: ignore
        "family_history": "1",
        "benefits": "Yes",
        "care_options": "Yes"
    },
    show_progress=False
)

print(q4)

+--------------+------------------+
| treatment    |   phi(treatment) |
+==============+==================+
| treatment(0) |           0.2535 |
+--------------+------------------+
| treatment(1) |           0.7465 |
+--------------+------------------+


In [18]:
# Query 5:
# How does work interference change under social/workplace conditions?
q5 = infer_restricted.query(
    variables=["work_interfere"],
    evidence={ # type: ignore
        "mental_health_consequence": "Yes",
        "supervisor": "No",
        "coworkers": "No"
    },
    show_progress=False
)

print(q5)

+---------------------------+-----------------------+
| work_interfere            |   phi(work_interfere) |
+===========================+=======================+
| work_interfere(Never)     |                0.2416 |
+---------------------------+-----------------------+
| work_interfere(Often)     |                0.1534 |
+---------------------------+-----------------------+
| work_interfere(Rarely)    |                0.1513 |
+---------------------------+-----------------------+
| work_interfere(Sometimes) |                0.4537 |
+---------------------------+-----------------------+


In [19]:
# Summarize the learned local neighborhood of the two key outcomes
# This gives a compact interpretation of the network structure
summary_rows = []

for target in ["treatment", "work_interfere"]:
    if target in bn_restricted.nodes():
        summary_rows.append({
            "Target": target,
            "Parents": list(bn_restricted.predecessors(target)),
            "Children": list(bn_restricted.successors(target))
        })

pd.DataFrame(summary_rows)

,Target,Parents,Children
0,treatment,"[Gender, work_interfere]",[]
1,work_interfere,"[Gender, family_history]","[no_employees, treatment, leave, care_options,..."
